In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
import seaborn as sns
import scipy
from os import path
from statsmodels import robust
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from skimage import img_as_uint
from skimage.io import imshow, imread
from skimage.color import rgb2hsv
from skimage.color import rgb2gray
import skimage as ski
from sklearn.metrics import balanced_accuracy_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow import keras
import os
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, UpSampling2D, concatenate, Concatenate, BatchNormalization
from keras.callbacks import EarlyStopping
from collections import Counter
import keras
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Concatenate, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import class_weight
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, balanced_accuracy_score
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

O modelo desenvolvido aceita dados em estrutura 2D (imagens) com o tamanho específico 256x256. Desta forma, todas as imagens que forem dadas como input ao modelo necessitam de ser redimensionadas (mantendo proporção) e "padded" para atingir o tamanho alvo.

# Training dataset preparation

In [ ]:
###downsample and pad (CORRER)
import os
import cv2
import re
import numpy as np

#função que prepara as imagens para treinar o modelo
def resize_and_pad(img, target_size): #recebe uma imagem e o tamanho alvo (h,w)
    """Redimensiona mantendo proporção e aplica padding para atingir o tamanho alvo"""
    h, w = img.shape #obtém-se a altura (h) e largura (w) da imagem atual (dimensões originais)
    target_h, target_w = target_size #atrui-se cada dimensão alvo a uma variável   h--->>>target_h e w--->>>target_w

    scale = min(target_w / w, target_h / h) 
    new_w, new_h = int(w * scale), int(h * scale)
    
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # Padding
    top = (target_h - new_h) // 2
    bottom = target_h - new_h - top
    left = (target_w - new_w) // 2
    right = target_w - new_w - left

    padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
    return padded


##visualização [ORIGINAL, DOWNSAMPLED, FINAL]
def visualize_processing_pipeline(original_img, downsample_factor=0.5, target_size=(384, 512)):
    # Step 1: Original
    original = original_img

    # Step 2: Downsample
    downsampled = cv2.resize(original, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_LINEAR)

    # Step 3: Resize and pad
    padded = resize_and_pad(downsampled, target_size)
    #padded= (padded > 0).astype("float32")

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    titles = ['Original', f'Downsampled (factor={downsample_factor})', f'Final (padded to {target_size})']
    images = [original, downsampled, padded]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

'''
base_path= os.getcwd() + "\\dataset_unet\\vol4\\octs_final\\slice_013.tif"
base_path2= os.getcwd() + "\\dataset_unet\\vol4\\gt_final\\slice_13.png"
img = cv2.imread(base_path, cv2.IMREAD_GRAYSCALE)
gt = cv2.imread(base_path2, cv2.IMREAD_GRAYSCALE)

visualize_processing_pipeline(img, downsample_factor=0.2, target_size=(256, 256))
visualize_processing_pipeline(gt, downsample_factor=0.2, target_size=(256, 256))
'''

In [ ]:
#CORRER
import os
import cv2
import re
import numpy as np

def natural_key(filename):
    """Ordena com base nos números dentro do nome do ficheiro"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', filename)]

#carregar volume e mask ground truth
def load_volume_and_mask(patient_folder, img_size=(256, 256), downsample_factor=1.0):
    slice_folder = os.path.join(patient_folder, 'octs_final')
    mask_folder = os.path.join(patient_folder, 'gt_final')

    slice_files = sorted([f for f in os.listdir(slice_folder) if f.endswith(('.tif', '.png'))], key=natural_key)
    mask_files = sorted([f for f in os.listdir(mask_folder) if f.endswith('.png')], key=natural_key)

    slices = []
    masks = []

    for slice_file, mask_file in zip(slice_files, mask_files):
        slice_path = os.path.join(slice_folder, slice_file)
        mask_path = os.path.join(mask_folder, mask_file)

        img = cv2.imread(slice_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # Downsampling proporcional
        img = cv2.resize(img, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_NEAREST)

        # Pad to fixed size
        img = resize_and_pad(img, img_size)
        mask = resize_and_pad(mask, img_size)
        #mask= (mask > 0).astype("float32")
        

        slices.append(img)
        masks.append(mask)

    return slices, masks

#visualização imagem vs mask
def visualize_image_and_mask(image, mask, title="Imagem e Máscara"):
    """
    Mostra lado a lado uma imagem e a sua máscara correspondente.
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(image.squeeze(), cmap='gray')
    axes[0].set_title("Imagem (X)")
    axes[0].axis('off')

    axes[1].imshow(mask.squeeze(), cmap='gray')
    axes[1].set_title("Máscara (Y)")
    axes[1].axis('off')

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# carregar DATASET PARA TREINAR O MODELO
# Carregar todas as slices (sem separação por volume)
X_slices, Y_slices = [], []
base_path= os.getcwd() + "\\dataset_unet"

for patient in sorted(os.listdir(base_path)):
    slice_imgs, mask_imgs = load_volume_and_mask(os.path.join(base_path, patient))

    for img, mask in zip(slice_imgs, mask_imgs):
        X_slices.append(img[..., np.newaxis] / 255.0)
        Y_slices.append(mask[..., np.newaxis] / 255.0)

X = np.array(X_slices)  # shape: (total_slices, height, width, 1)
Y = np.array(Y_slices)  # shape: (total_slices, height, width, 1)

print("Shape de X:", X.shape)
print("Shape de Y:", Y.shape)

In [ ]:
for i in range(802):  
    visualize_image_and_mask(X[i], Y[i], title=f"Slice {i}")

In [ ]:
#carregar só um volume de teste (na ausência de masks ground truth)
def load_test_volume(test_folder, img_size=(256, 256), downsample_factor=1.0):
    """
    Carrega apenas as imagens de um volume (sem máscaras) para dados de teste.

    Args:
        test_folder (str): Caminho para a pasta do paciente contendo a subpasta 'octs_final'.
        img_size (tuple): Tamanho final da imagem após padding.
        downsample_factor (float): Fator de downsampling.

    Returns:
        slices (list): Lista de imagens processadas.
    """
    #slice_folder = os.path.join(test_folder, 'octs_final') comentei por perguiça 
    slice_folder = os.path.join(test_folder)

    slice_files = sorted(
        [f for f in os.listdir(slice_folder) if f.endswith(('.tif', '.png'))],
        key=natural_key
    )

    slices = []

    for slice_file in slice_files:
        slice_path = os.path.join(slice_folder, slice_file)

        img = cv2.imread(slice_path, cv2.IMREAD_GRAYSCALE)

        # Downsampling proporcional
        img = cv2.resize(img, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_LINEAR)

        # Pad to fixed size
        img = resize_and_pad(img, img_size)

        slices.append(img)

    return slices

In [ ]:
#carregar vários volumes (só images), neste caso os do porto e antes de ter as masks ground truth
X_slices_porto= []
base_path= os.getcwd() + "\\dataset_porto"

for patient in sorted(os.listdir(base_path)):
    slice_imgs= load_test_volume(os.path.join(base_path, patient))

    for img in slice_imgs:
        X_slices_porto.append(img[..., np.newaxis] / 255.0)
        #Y_slices.append(mask[..., np.newaxis] / 255.0)

X_porto = np.array(X_slices_porto)  # shape: (total_slices, height, width, 1)
#Y = np.array(Y_slices)  # shape: (total_slices, height, width, 1)

print("Shape de X:", X_porto.shape)
#print("Shape de Y:", Y.shape)

In [ ]:
#fazer o teste para os volumes do porto 
X_slices_porto, Y_slices_porto = [], []

# Define o caminho para um volume específico
base_path= os.getcwd() + "\\dataset_porto"

for patient in sorted(os.listdir(base_path)):
    slice_imgs, mask_imgs = load_volume_and_mask(os.path.join(base_path, patient))

    for img, mask in zip(slice_imgs, mask_imgs):
        X_slices_porto.append(img[..., np.newaxis] / 255.0)  # normaliza
        Y_slices_porto.append(mask[..., np.newaxis] / 255.0)  # normaliza


# Converte para arrays
Xtest_porto = np.array(X_slices_porto)
Ytest_porto = np.array(Y_slices_porto)

print("Shape de Xtest_porto:", Xtest_porto.shape)
print("Shape de Ytest_porto:", Ytest_porto.shape)

In [ ]:
# Binarização das máscaras
Ytest_porto = (Ytest_porto >=0.5).astype("float32")

print(np.unique(Ytest_porto)) 


In [ ]:
#só para treinar o modelo B e testar as metricas
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.20, random_state=42)
print(y_train.shape)
print(X_train.shape)
print(y_test.shape)
print(X_test.shape)

In [ ]:
# Binarização das máscaras
#y_train = (y_train >= 0.5).astype("float32")
#y_val = (y_val >= 0.5).astype("float32")
y_train = (y_train >0).astype("float32")
#y_val = (y_val >0).astype("float32")
y_test= (y_test >0).astype("float32")

print(np.unique(y_train)) 
#print(np.unique(y_val))# Deve dar [0. 1.]
#print(X_val.shape)

# Evaluation Metrics + Dataset Augmentation + Model 

In [ ]:
#métricas
import tensorflow.keras.backend as K

def dice_coefficient(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

import tensorflow as tf
from tensorflow.keras import backend as K

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection

    return (intersection + smooth) / (union + smooth)



In [ ]:
#data augmentation antes de treinar o modelo
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Geradores com as MESMAS transformações
datagen = ImageDataGenerator(rotation_range=5, 
                             width_shift_range=0.02,
                             height_shift_range=0.1,
                             zoom_range=0.05,
                             horizontal_flip=True,
                             fill_mode='constant',
                             cval=0)


#Mudar o batch_size caso necessário
seed = 42
batch_size = 32

image_generator = datagen.flow(X_train, batch_size=batch_size, seed=seed)
mask_generator  = datagen.flow(y_train, batch_size=batch_size, seed=seed)

# Gerador combinado (zip os dois geradores)
#train_generator = zip(image_generator, mask_generator) #NOTA: Isto do zip n funciona

def combined_generator(image_gen, mask_gen):
    while True:
        X = next(image_gen)
        y = next(mask_gen)
        yield X, y

train_generator = combined_generator(image_generator, mask_generator)

import matplotlib.pyplot as plt

images_aug, masks_aug = next(train_generator)

# nº de amostras a visualizar
n = 5

plt.figure(figsize=(12, 5))
for i in range(n):
    # Imagem
    plt.subplot(2, n, i + 1)
    plt.imshow(images_aug[i].squeeze(), cmap='gray')
    plt.title("Imagem")
    plt.axis('off')
    
    # Máscara
    plt.subplot(2, n, i + 1 + n)
    plt.imshow(masks_aug[i].squeeze(), cmap='gray')
    plt.title("Máscara")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#modelo final
from tensorflow.keras import layers, models
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Dropout, UpSampling2D,
    Concatenate, Cropping2D, Activation, Multiply, Add, Lambda
)
from keras.layers import GroupNormalization
import tensorflow as tf

# Crop to Match 
def crop_to_match(skip, up):
    sh, sw = skip.shape[1], skip.shape[2]
    uh, uw = up.shape[1], up.shape[2]
    crop_h = sh - uh
    crop_w = sw - uw

    crop_top = crop_h // 2
    crop_bottom = crop_h - crop_top
    crop_left = crop_w // 2
    crop_right = crop_w - crop_left

    return Cropping2D(((crop_top, crop_bottom), (crop_left, crop_right)))(skip)

# Attention Gate
def attention_gate(x, g, inter_channels):
    x = crop_to_match(x, g)

    theta_x = Conv2D(inter_channels, 1, padding='same')(x)
    phi_g = Conv2D(inter_channels, 1, padding='same')(g)
    add = Add()([theta_x, phi_g])
    relu = Activation('relu')(add)

    psi = Conv2D(1, 1, padding='same')(relu)
    sigmoid = Activation('sigmoid')(psi)

    # Funções auxiliares
    def repeat_channels(s, n_channels):
        return tf.repeat(s, repeats=n_channels, axis=-1)

    def repeat_channels_output_shape(input_shape, n_channels):
        return input_shape[:-1] + (n_channels,)

    # Usa Lambda com output_shape definido
    sigmoid_expanded = Lambda(
        repeat_channels,
        output_shape=lambda s: repeat_channels_output_shape(s, x.shape[-1]),
        arguments={'n_channels': x.shape[-1]}
    )(sigmoid)

    gated = Multiply()([x, sigmoid_expanded])

    out = Conv2D(x.shape[-1], 1, padding='same')(gated)
    out = GroupNormalization(groups=16)(out)
    return out


# U-Net with GroupNormalization 
def unet_attention_model3(input_shape):
    inputs = Input(shape=input_shape)

    # Encoder 
    c1 = Conv2D(32, 3, padding='same')(inputs)
    c1 = GroupNormalization(groups=16)(c1)
    c1 = Activation('relu')(c1)
    c1 = Conv2D(32, 3, padding='same')(c1)
    c1 = GroupNormalization(groups=16)(c1)
    c1 = Activation('relu')(c1)
    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(64, 3, padding='same')(p1)
    c2 = GroupNormalization(groups=16)(c2)
    c2 = Activation('relu')(c2)
    c2 = Conv2D(64, 3, padding='same')(c2)
    c2 = GroupNormalization(groups=16)(c2)
    c2 = Activation('relu')(c2)
    c2 = Dropout(0.2)(c2)
    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(128, 3, padding='same')(p2)
    c3 = GroupNormalization(groups=16)(c3)
    c3 = Activation('relu')(c3)
    c3 = Conv2D(128, 3, padding='same')(c3) 
    c3 = GroupNormalization(groups=16)(c3)
    c3 = Activation('relu')(c3)
    c3 = Dropout(0.3)(c3)
    p3 = MaxPooling2D()(c3)

    # Bottleneck 
    c4 = Conv2D(256, 3, padding='same')(p3)
    c4 = GroupNormalization(groups=16)(c4)
    c4 = Activation('relu')(c4)
    c4 = Conv2D(256, 3, padding='same')(c4)
    c4 = GroupNormalization(groups=16)(c4)
    c4 = Activation('relu')(c4)
    c4 = Dropout(0.4)(c4)

    # Decoder 
    u5 = UpSampling2D()(c4)
    att3 = attention_gate(c3, u5, inter_channels=128) 
    u5 = Concatenate()([u5, att3])
    c5 = Conv2D(128, 3, padding='same')(u5)
    c5 = GroupNormalization(groups=16)(c5)
    c5 = Activation('relu')(c5)
    c5 = Conv2D(128, 3, padding='same')(c5)
    c5 = GroupNormalization(groups=16)(c5)
    c5 = Activation('relu')(c5)
    c5 = Dropout(0.3)(c5)

    u6 = UpSampling2D()(c5)
    att2 = attention_gate(c2, u6, inter_channels=64) 
    u6 = Concatenate()([u6, att2])
    c6 = Conv2D(64, 3, padding='same')(u6)
    c6 = GroupNormalization(groups=16)(c6)
    c6 = Activation('relu')(c6)
    c6 = Conv2D(64, 3, padding='same')(c6)
    c6 = GroupNormalization(groups=16)(c6)
    c6 = Activation('relu')(c6)
    c6 = Dropout(0.2)(c6)

    u7 = UpSampling2D()(c6)
    att1 = attention_gate(c1, u7, inter_channels=32) 
    u7 = Concatenate()([u7, att1])
    c7 = Conv2D(32, 3, padding='same')(u7)
    c7 = GroupNormalization(groups=16)(c7)
    c7 = Activation('relu')(c7)
    c7 = Conv2D(32, 3, padding='same')(c7)
    c7 = GroupNormalization(groups=16)(c7)
    c7 = Activation('relu')(c7)

    outputs = Conv2D(1, 1, activation='sigmoid')(c7)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
#visualisar image vs ground truth mask vs predicted mask
def plot_all_predictions(X, y_true, y_pred):
    """
    Mostra previsões do modelo para todas as amostras em X.
    
    Args:
        X: imagens originais (shape: [N, H, W, C])
        y_true: máscaras verdadeiras (shape: [N, H, W, 1])
        y_pred: máscaras previstas (shape: [N, H, W, 1])
    """
    num_samples = len(X)

    plt.figure(figsize=(12, num_samples * 4))
    for i in range(num_samples):
        plt.subplot(num_samples, 3, i * 3 + 1)
        plt.imshow(X[i].squeeze(), cmap="gray")
        plt.title(f"Original (idx={i})")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 2)
        plt.imshow(y_true[i].squeeze(), cmap="gray")
        plt.title("True mask")
        plt.axis("off")

        plt.subplot(num_samples, 3, i * 3 + 3)
        plt.imshow(y_pred[i].squeeze(), cmap="gray")
        plt.title("Predicted mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

#visualizar image vs predicted mask
def plot_predictions2(X, y_pred, num_samples=10):
    plt.figure(figsize=(8, num_samples * 4))
    
    for i in range(num_samples):
        # Imagem original
        plt.subplot(num_samples, 2, i * 2 + 1)
        plt.imshow(X[i].squeeze(), cmap="gray")
        plt.title("Imagem")
        plt.axis("off")

        # Máscara prevista
        plt.subplot(num_samples, 2, i * 2 + 2)
        plt.imshow(y_pred[i].squeeze(), cmap="gray")
        plt.title("Máscara prevista")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    

# Load previously trained models

In [ ]:
#carregar modelos já guardados -> model_augfin4 modelo final da tese
model_augfin4 = unet_attention_model3(input_shape=(256, 256, 1))  # ou o shape que usaste
model_augfin4.compile(optimizer='adam', loss=dice_loss, metrics=[dice_coefficient, iou_metric])
model_augfin4.load_weights("model_augfin4.weights.h5")

In [ ]:
#carregar modelos já guardados -> model_augfin5 modelo treinado depois da tese com mais imagens
model_augfin5 = unet_attention_model3(input_shape=(256, 256, 1))  # ou o shape que usaste
model_augfin5.compile(optimizer='adam', loss=dice_loss, metrics=[dice_coefficient, iou_metric])
model_augfin5.load_weights("model_augfin5.weights.h5")

In [ ]:
y_test_pred = (model_augfin4.predict(X_test) > 0.5).astype("int32")

In [ ]:
#NÃO CORRER; serve apenas para análise
# Filtrar apenas imagens com pelo menos 1 pixel positivo no ground truth 
mask_non_empty = np.array([np.sum(y) > 0 for y in y_test])

X_test_nonempty = X_test[mask_non_empty]
y_test_nonempty = y_test[mask_non_empty]
y_test_pred_nonempty = y_test_pred[mask_non_empty]
print(y_test_nonempty.shape)

print(f"Imagens com lesão: {np.sum(mask_non_empty)} / {len(y_test)}")

plot_all_predictions(X_test_nonempty, y_test_nonempty, y_test_pred_nonempty)

In [ ]:
#NÃO CORRER; serve apenas para análise
mask_empty = np.array([np.sum(y) == 0 for y in y_test])

X_test_empty = X_test[mask_empty]
y_test_empty = y_test[mask_empty]
y_test_pred_empty = y_test_pred[mask_empty]
print(y_test_empty.shape)

print(f"Imagens sem lesão: {np.sum(mask_empty)} / {len(y_test)}")

plot_all_predictions(X_test_empty, y_test_empty, y_test_pred_empty)

In [ ]:
#Análise quantitativa (métricas, histogramas, boxplots) 
import numpy as np
import matplotlib.pyplot as plt
import os

def dice_coefficient_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.astype("float32").flatten()
    y_pred_f = y_pred.astype("float32").flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def sensitivity_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    TP = np.sum(y_true_f * y_pred_f)
    FN = np.sum(y_true_f * (1 - y_pred_f))
    return (TP + smooth) / (TP + FN + smooth)

def specificity_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    TN = np.sum((1 - y_true_f) * (1 - y_pred_f))
    FP = np.sum((1 - y_true_f) * y_pred_f)
    return (TN + smooth) / (TN + FP + smooth)

def balanced_accuracy_np(y_true, y_pred, smooth=1e-6):
    sens = sensitivity_np(y_true, y_pred, smooth)
    spec = specificity_np(y_true, y_pred, smooth)
    return (sens + spec) / 2

# Compute metrics for each image 
metrics = {
    "DSC": [],
    "IoU": [],
    "Sensitivity": [],
    "Specificity": [],
    "Balanced Accuracy": []
}

for i in range(len(y_test)):
    y_t = y_test[i]
    y_p = y_test_pred[i]

    metrics["DSC"].append(dice_coefficient_np(y_t, y_p) * 100)
    metrics["IoU"].append(iou_np(y_t, y_p) * 100)
    metrics["Sensitivity"].append(sensitivity_np(y_t, y_p) * 100)
    metrics["Specificity"].append(specificity_np(y_t, y_p) * 100)
    metrics["Balanced Accuracy"].append(balanced_accuracy_np(y_t, y_p) * 100)

# Convert to numpy arrays
for k in metrics.keys():
    metrics[k] = np.array(metrics[k])

# Create folder to save plots 
output_dir = "results_plots_todas_red_percentagem"
os.makedirs(output_dir, exist_ok=True)

# Boxplots
#plt.figure(figsize=(6,6))
#plt.boxplot([metrics[m] for m in metrics.keys()], labels=list(metrics.keys()))
#plt.ylabel("Accuracy (%)")
#plt.grid(True, linestyle="--", alpha=0.7)
#plt.tight_layout()
#plt.savefig(os.path.join(output_dir, "boxplot_metrics.png"), dpi=300, bbox_inches="tight")
#plt.show()

# Histograms 
for m in metrics.keys():
    plt.figure(figsize=(4,4))
    plt.hist(metrics[m], bins=20, edgecolor="black", alpha=0.7, color="red")
    plt.title(f"Distribution of {m}")
    plt.xlabel(f"{m} (%)")
    plt.ylabel("Number of images")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()

    # Save histogram
    #filename = f"hist_{m.replace(' ', '_')}.png"
    #plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    #plt.close()  # close to save memory

    #print(f"Histogram for {m} saved as {os.path.join(output_dir, filename)}")

# Statistics
for m in metrics.keys():
    print(f"{m}: Mean = {metrics[m].mean():.2f}%, Std. Dev. = {metrics[m].std():.2f}%, Median= {np.median(metrics[m]):.2f}%, IQR (Q3 - Q1): {np.percentile(metrics[m],75) - np.percentile(metrics[m],25):.4f}%")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# --- Folder to save plots ---
output_dir = "results_boxplots_per_metric_percent"
os.makedirs(output_dir, exist_ok=True)

# --- Boxplot for each metric ---
for m in metrics.keys():
    plt.figure(figsize=(4, 4))
    plt.boxplot(metrics[m], labels=[m])
    plt.ylabel("Accuracy (%)")  # ← mostra que é percentagem
    plt.title(f"Boxplot of {m} (%)")  # ← título também em %
    plt.grid(True, linestyle="--", alpha=0.7)
    #plt.ylim(0, 100)  # ← opcional: força o eixo Y a 0–100%
    plt.tight_layout()
    
    # Save the plot
    #filename = f"boxplot_{m.replace(' ', '_')}.png"
    #plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    #plt.close()
    
    #print(f"Saved boxplot for {m} → {filename}")


import numpy as np
import matplotlib.pyplot as plt

areas_gt = []
areas_pred = []
relative_deviations = []

for i in range(len(y_test)):
    y_t = y_test[i]
    y_p = y_test_pred[i]

    # --- Areas (in pixels) ---
    area_gt = np.sum(y_t)
    area_pred = np.sum(y_p)
    areas_gt.append(area_gt)
    areas_pred.append(area_pred)

    # --- Relative deviation (%)
    if area_gt > 0:
        rel_dev = 100 * (area_pred - area_gt) / area_gt
    else:
        rel_dev = np.nan
    relative_deviations.append(rel_dev)

areas_gt = np.array(areas_gt)
areas_pred = np.array(areas_pred)
relative_deviations = np.array(relative_deviations)
valid_rel_dev = relative_deviations[~np.isnan(relative_deviations)]

# --- Statistics ---
print("\n=== Relative Area Deviation Statistics ===")
print(f"Mean relative deviation: {np.mean(valid_rel_dev):.4f}%")
print(f"Standard deviation: {np.std(valid_rel_dev):.4f}%")
print(f"Median: {np.median(valid_rel_dev):.4f}%")
print(f"IQR (Q3 - Q1): {np.percentile(valid_rel_dev, 75) - np.percentile(valid_rel_dev, 25):.4f}%")

plt.figure(figsize=(4,5))
#plt.boxplot(valid_rel_dev, vert=True, widths=0.6)
plt.boxplot([valid_rel_dev], labels=["Relative deviation"], vert=True, widths=0.6)
plt.ylabel("Relative deviation (%)")
#plt.title("Distribution of relative area deviations")
plt.xticks([])  # remove the "1" on the x-axis
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
#plt.savefig("boxplot_dev.png", dpi=300, bbox_inches="tight")
plt.show()


# --- Histogram of relative deviation ---
plt.figure(figsize=(6,4))
plt.hist(valid_rel_dev, bins=20, edgecolor="black", alpha=0.7, color="red")
plt.xlabel("Relative deviation (%)")
plt.ylabel("Number of images")
plt.title("Distribution of Relative Area Deviations")
plt.grid(True, linestyle="--", alpha=0.7)
#plt.savefig("histogram_dev.png", dpi=300, bbox_inches="tight")
plt.show()


import matplotlib.pyplot as plt
import numpy as np

# Exemplo com os teus dados
data = np.array(valid_rel_dev)

# Cria duas subplots lado a lado (eixo X partido)
fig, (ax_left, ax_right) = plt.subplots(
    1, 2, sharey=True, figsize=(4, 4), gridspec_kw={'width_ratios': [1, 1]}
)

# --- Histograma principal (valores bons) ---
ax_left.hist(data, bins=20, color="red", edgecolor="black", alpha=0.7)
plt.title("Distribution of Relative Area Deviations")
ax_left.set_xlim(-75, 85)  # zona central
ax_left.grid(True, linestyle="--", alpha=0.7)
ax_left.set_ylabel("Number of images")
ax_left.set_xlabel("Relative area deviation (%)")

# --- Histograma dos outliers (valores extremos) ---
ax_right.hist(data, bins=20, color="red", edgecolor="black", alpha=0.7)
ax_right.set_xlim(390, 450)  # zona com valores extremos
ax_right.grid(True, linestyle="--", alpha=0.7)
#ax_right.set_xlabel("Relative deviation (%)")

# --- Quebra visual (diagonais no eixo) ---
ax_left.spines['right'].set_visible(False)
ax_right.spines['left'].set_visible(False)
ax_right.yaxis.tick_right()

d = .015  # tamanho da diagonal
kwargs = dict(transform=ax_left.transAxes, color='k', clip_on=False)
ax_left.plot((1 - d, 1 + d), (-d, +d), **kwargs)
ax_left.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

kwargs.update(transform=ax_right.transAxes)
ax_right.plot((-d, +d), (-d, +d), **kwargs)
ax_right.plot((-d, +d), (1 - d, 1 + d), **kwargs)

plt.tight_layout()
#plt.savefig("histogram_dev_broken.png", dpi=300, bbox_inches="tight")
plt.show()

fig, (ax_top, ax_bottom) = plt.subplots(
    2, 1, sharex=True, figsize=(4, 4), gridspec_kw={'height_ratios': [1, 2]}
)

# Upper and lower boxplots (same data)
ax_top.boxplot([valid_rel_dev], widths=0.6)
ax_bottom.boxplot([valid_rel_dev], widths=0.6)

# Focus ranges
ax_top.set_ylim(390, 450)
ax_bottom.set_ylim(-75, 75)

# Hide the middle space (simulate break)
ax_top.spines['bottom'].set_visible(False)
ax_bottom.spines['top'].set_visible(False)
ax_top.tick_params(bottom=False)

# Add diagonal lines to mark the break
d = .015
kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

kwargs.update(transform=ax_bottom.transAxes)
ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

ax_top.grid(True, linestyle="--", alpha=0.7)
#ax_top.set_ylabel("Relative deviation (%)")

ax_bottom.grid(True, linestyle="--", alpha=0.7)
ax_bottom.set_ylabel("Metric value (%)")
ax_bottom.set_xlabel("Relative area deviation")

#ax_bottom.set_ylabel("Relative deviation (%)")

fig.suptitle("Distribution of relative area deviations", fontsize=12, y=0.94)
plt.xticks([])
plt.tight_layout()
#plt.ylabel("Relative deviation (%)")
#plt.title("Distribution of relative area deviations")
#plt.xticks([])  # remove the "1" on the x-axis
plt.grid(True, linestyle="--", alpha=0.7)
#plt.tight_layout()
#plt.savefig("boxplot_dev_broken_matplotlib.png", dpi=300, bbox_inches="tight")
plt.show()


retinal_detach_sizes = [np.sum(y) for y in y_test]
retinal_detach_sizes = np.array(retinal_detach_sizes)

plt.figure(figsize=(1,5))
plt.scatter(retinal_detach_sizes, metrics["DSC"], color="blue", alpha=0.5, label="DSC")
plt.scatter(retinal_detach_sizes, metrics["IoU"], color="green", alpha=0.5, label="IoU")
plt.scatter(retinal_detach_sizes, metrics["Balanced Accuracy"], color="pink", alpha=0.5, label="Balanced Accuracy")

plt.xlabel("Detachment size (pixels)")
plt.ylabel("Accuracy (%)")  # ← mostra que é percentagem
plt.title("Relationship between retinal detachment size and segmentation accuracy (%)")  # ← título ajustado
#plt.ylim(0, 100)  # ← opcional: força o eixo Y a ir de 0 a 100%
plt.grid(True, linestyle="--", alpha=0.7)
plt.legend()
plt.tight_layout()

#plt.savefig("scatter_metrics_vs_size_percent.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
from scipy import stats

def confidence_interval(data, confidence=0.95):
    mean = np.mean(data)
    sem = stats.sem(data)  # erro padrão da média
    h = sem * stats.t.ppf((1 + confidence) / 2., len(data)-1)
    return mean, mean-h, mean+h

for m in metrics.keys():
    mean, ci_low, ci_high = confidence_interval(metrics[m])
    print(f"{m}: Mean = {mean:.4f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

import numpy as np

valid_rel_dev = np.array(valid_rel_dev)
mean = np.mean(valid_rel_dev)
std = np.std(valid_rel_dev, ddof=1)
n = len(valid_rel_dev)

ci95_low = mean - 1.96 * std / np.sqrt(n)
ci95_high = mean + 1.96 * std / np.sqrt(n)

print(f"95% CI for mean relative deviation: [{ci95_low:.4f}%, {ci95_high:.4f}%]")


In [ ]:
#precision recal curve

from sklearn.metrics import precision_recall_curve, average_precision_score

y_true_flat = y_test.flatten()
y_prob_flat = y_test_pred_prob.flatten()

precision, recall, thresholds = precision_recall_curve(y_true_flat, y_prob_flat)
ap = average_precision_score(y_true_flat, y_prob_flat)

print(f"Average Precision (AP) = {ap:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
plt.plot(recall, precision, linewidth=2)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(True)

plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# --- Flatten all ground truth and prediction arrays ---
# Ensure they are binary masks (0 or 1)
y_true_all = np.concatenate([y.flatten() for y in y_test])
y_pred_all = np.concatenate([y.flatten() for y in y_test_pred])

# Just in case values are not 0/1 exactly (e.g. float probabilities)
#y_true_all = (y_true_all > 0.5).astype(int)
#y_pred_all = (y_pred_all > 0.5).astype(int)

# --- Compute confusion matrix ---
cm = confusion_matrix(y_true_all, y_pred_all)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Background", "Detachment"])

# --- Plot ---
disp.plot(cmap=plt.cm.Purples, values_format='d')
plt.title("Global pixel-wise confusion matrix (all test images)")
plt.savefig("confusion_matrix_global.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Extract values ---
tn, fp, fn, tp = cm.ravel()

# --- Compute metrics ---
sensitivity = tp / (tp + fn + 1e-8)
specificity = tn / (tn + fp + 1e-8)
balanced_acc = (sensitivity + specificity) / 2

print(f"TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
print(f"Sensitivity (Recall): {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")

In [ ]:
# --- Global relative deviation ---
total_area_gt = np.nansum(areas_gt)
total_area_pred = np.nansum(areas_pred)

global_rel_dev = 100 * (total_area_pred - total_area_gt) / total_area_gt

print(f"\nGlobal relative deviation: {global_rel_dev:.4f}%")

# Flatten all predictions and truths
y_true_all = np.concatenate([y.flatten() for y in y_test_empty])
y_pred_all = np.concatenate([y.flatten() for y in y_test_pred_empty])

# Compute metrics globally
print("Global Dice:", dice_coefficient_np(y_true_all, y_pred_all))
print("Global IoU:", iou_np(y_true_all, y_pred_all))
print("Global Sensitivity:", sensitivity_np(y_true_all, y_pred_all))
print("Global Specificity:", specificity_np(y_true_all, y_pred_all))
print("Global Balanced Accuracy:", balanced_accuracy_np(y_true_all, y_pred_all))
#print("Global Relative Deviation:", balanced_accuracy_np(y_true_all, y_pred_all))

y_true_all = np.concatenate([y.flatten() for y in y_test_nonempty])
y_pred_all = np.concatenate([y.flatten() for y in y_test_pred_nonempty])

# Compute metrics globally
print("Global Dice:", dice_coefficient_np(y_true_all, y_pred_all))
print("Global IoU:", iou_np(y_true_all, y_pred_all))
print("Global Sensitivity:", sensitivity_np(y_true_all, y_pred_all))
print("Global Specificity:", specificity_np(y_true_all, y_pred_all))
print("Global Balanced Accuracy:", balanced_accuracy_np(y_true_all, y_pred_all))
#print("Global Relative Deviation:", balanced_accuracy_np(y_true_all, y_pred_all))

# Flatten all predictions and truths
y_true_all = np.concatenate([y.flatten() for y in y_test])
y_pred_all = np.concatenate([y.flatten() for y in y_test_pred])

# Compute metrics globally
dice = dice_coefficient_np(y_true_all, y_pred_all)
iou = iou_np(y_true_all, y_pred_all)
sens = sensitivity_np(y_true_all, y_pred_all)
spec = specificity_np(y_true_all, y_pred_all)
bal_acc = balanced_accuracy_np(y_true_all, y_pred_all)

# Print formatted metrics (4 decimal places)
print(f"Global Dice: {dice:.4f}")
print(f"Global IoU: {iou:.4f}")
print(f"Global Sensitivity: {sens:.4f}")
print(f"Global Specificity: {spec:.4f}")
print(f"Global Balanced Accuracy: {bal_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np

def plot_error_full(gt, pred, error_rgb, i):
    """
    Mostra GT, predição e mapa de erros no tamanho original (sem zoom).
    """
    gt = np.squeeze(gt)
    pred = np.squeeze(pred)
    error_rgb = np.squeeze(error_rgb)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(gt, cmap="gray")
    plt.title(f"GT idx={i}")
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(pred, cmap="gray")
    plt.title(f"Predição")
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(error_rgb)
    plt.title(f"Mapa de erros")
    plt.axis('off')

    plt.tight_layout()
    plt.show()


import numpy as np

def make_error_map_slice(gt_bin_slice, pred_bin_slice, retina_mask_slice=None):
    """
    Cria mapa RGB de erros para uma slice.

    Args:
        gt_bin_slice (2D array ou 3D com último canal 1): ground truth binário (0/1)
        pred_bin_slice (2D array ou 3D com último canal 1): predição binária (0/1)
        retina_mask_slice (2D array, opcional): máscara retina (0/1)
    """
    gt = np.squeeze(gt_bin_slice).astype(bool)
    pr = np.squeeze(pred_bin_slice).astype(bool)

    tp = (gt & pr)
    fp = (~gt & pr)
    fn = (gt & ~pr)

    if retina_mask_slice is not None:
        mask = np.squeeze(retina_mask_slice).astype(bool)
        tp = tp & mask
        fp = fp & mask
        fn = fn & mask

    h, w = gt.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    out[tp] = [128, 128, 128]   # cinza
    out[fp] = [255, 0, 0]   # vermelho
    out[fn] = [0, 0, 255]   # azul

    return out, tp.sum(), fp.sum(), fn.sum()

'''
for i in range(len(y_test)):
    err_rgb, n_tp, n_fp, n_fn = make_error_map_slice(y_test[i], y_test_pred[i])
    print("TP:", n_tp, "FP:", n_fp, "FN:", n_fn)
    plot_error_full(y_test[i], y_test_pred[i], err_rgb, i)
'''

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_overlay_predictions(X_test, y_test_pred, num_samples=None, save_dir=None,
                             alpha=0.4, color='red'):
    """
    Overlay only the predicted detachment area on the OCT images (solid color overlay).

    Args:
        X_test (np.ndarray): OCT images (N,H,W) or (N,H,W,1)
        y_test_pred (np.ndarray): predicted masks (N,H,W) or (N,H,W,1)
        num_samples (int): number of samples to show (default: all)
        save_dir (str): optional directory to save the overlay images
        alpha (float): transparency of overlay mask
        color (str): overlay color ('red', 'green', 'blue', 'yellow', etc.)
    """
    X_test = np.squeeze(X_test)
    y_test_pred = np.squeeze(y_test_pred)

    n = X_test.shape[0]
    if num_samples is None or num_samples > n:
        num_samples = n

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    # converter nome da cor para RGB (0–1 range)
    from matplotlib.colors import to_rgb
    overlay_color = np.array(to_rgb(color))

    for i in range(num_samples):
        img = X_test[i]
        mask = y_test_pred[i]

        mask_bin = (mask > 0.5).astype(float)

        # converter imagem grayscale para RGB
        img_rgb = np.stack([img]*3, axis=-1)

        # aplicar overlay colorido onde há máscara
        overlay = img_rgb.copy()
        overlay[mask_bin == 1] = (
            (1 - alpha) * overlay[mask_bin == 1] + alpha * overlay_color
        )

        plt.figure(figsize=(6, 6))
        plt.imshow(overlay)
        plt.title(f"OCT + Predicted Detachment {i} ({color})")
        plt.axis('off')
        plt.tight_layout()

        if save_dir:
            plt.savefig(os.path.join(save_dir, f"overlay_{i:03d}.png"),
                        bbox_inches='tight', pad_inches=0)
        else:
            plt.show()

        plt.close()

#plot_overlay_predictions(X_test, y_test_pred, num_samples=163, alpha=0.4, color='red')
#plot_overlay_predictions(X_porto, y_test_pred_porto, num_samples=601, alpha=0.4, color='red')

# Testing the model on a specific dataset - PORTO

In [ ]:
y_test_pred_porto_before = (model_augfin4.predict(Xtest_porto) > 0.5).astype("int32")

In [ ]:
y_test_pred_porto = (model_augfin5.predict(Xtest_porto) > 0.5).astype("int32")

In [ ]:
y_test_pred_porto2 = (model_augfin5.predict(Xtest_porto) > 0.6).astype("int32")

In [ ]:
y_test_pred_porto3_after = (model_augfin5.predict(Xtest_porto) > 0.6298415).astype("int32")


In [ ]:
#comparar novo modelo com este (antigo)
plot_overlay_predictions(Xtest_porto, y_test_pred_porto_before, num_samples=601, save_dir="overlays_porto_before", alpha=0.4, color='red')

In [ ]:
plot_overlay_predictions(Xtest_porto, y_test_pred_porto3_after, num_samples=601, save_dir="overlays_porto3_after", alpha=0.4, color='red')

In [ ]:
# --- Compute metrics for each image --- PARA O PORTO; metricas para model_augfin5

# --- Metric functions ---
def dice_coefficient_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.astype("float32").flatten()
    y_pred_f = y_pred.astype("float32").flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    intersection = np.sum(y_true_f * y_pred_f)
    union = np.sum(y_true_f) + np.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def sensitivity_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    TP = np.sum(y_true_f * y_pred_f)
    FN = np.sum(y_true_f * (1 - y_pred_f))
    return (TP + smooth) / (TP + FN + smooth)

def specificity_np(y_true, y_pred, smooth=1e-6):
    y_true_f = y_true.flatten()
    y_pred_f = y_pred.flatten()
    TN = np.sum((1 - y_true_f) * (1 - y_pred_f))
    FP = np.sum((1 - y_true_f) * y_pred_f)
    return (TN + smooth) / (TN + FP + smooth)

def balanced_accuracy_np(y_true, y_pred, smooth=1e-6):
    sens = sensitivity_np(y_true, y_pred, smooth)
    spec = specificity_np(y_true, y_pred, smooth)
    return (sens + spec) / 2

metrics = {
    "DSC": [],
    "IoU": [],
    "Sensitivity": [],
    "Specificity": [],
    "Balanced Accuracy": []
}

for i in range(len(Ytest_porto)):
    y_t = Ytest_porto[i]
    y_p = y_test_pred_porto3_after[i]

    metrics["DSC"].append(dice_coefficient_np(y_t, y_p) * 100)
    metrics["IoU"].append(iou_np(y_t, y_p) * 100)
    metrics["Sensitivity"].append(sensitivity_np(y_t, y_p) * 100)
    metrics["Specificity"].append(specificity_np(y_t, y_p) * 100)
    metrics["Balanced Accuracy"].append(balanced_accuracy_np(y_t, y_p) * 100)

# Convert to numpy arrays
for k in metrics.keys():
    metrics[k] = np.array(metrics[k])

# --- Create folder to save plots ---
output_dir = "results_plots_todas_red_percentagem_porto3_after"
os.makedirs(output_dir, exist_ok=True)


# --- Histograms ---
for m in metrics.keys():
    plt.figure(figsize=(4,4))
    plt.hist(metrics[m], bins=20, edgecolor="black", alpha=0.7, color="red")
    plt.title(f"Distribution of {m}")
    plt.xlabel(f"{m} (%)")
    plt.ylabel("Number of images")
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()

    # Save histogram
    #filename = f"hist_{m.replace(' ', '_')}.png"
    #plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    #plt.close()  # close to save memory

    #print(f"Histogram for {m} saved as {os.path.join(output_dir, filename)}")

# --- Statistics ---
for m in metrics.keys():
    print(f"{m}: Mean = {metrics[m].mean():.2f}%, Std. Dev. = {metrics[m].std():.2f}%, Median= {np.median(metrics[m]):.2f}%, IQR (Q3 - Q1): {np.percentile(metrics[m],75) - np.percentile(metrics[m],25):.4f}%")

In [ ]:
for m in metrics.keys():
    mean, ci_low, ci_high = confidence_interval(metrics[m])
    print(f"{m}: Mean = {mean:.4f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")

In [ ]:
#precision recal curve

from sklearn.metrics import precision_recall_curve, average_precision_score

#y_true_flat = Ytest_porto.flatten()
#y_prob_flat = y_test_pred_porto.flatten()

y_scores = model_augfin5.predict(Xtest_porto)

y_true_flat = Ytest_porto.flatten()
y_prob_flat = y_scores.flatten()

#precision, recall, _ = precision_recall_curve(y_test, y_scores)
precision, recall, thresholds = precision_recall_curve(y_true_flat, y_prob_flat)
#auc(recall, precision)
ap = average_precision_score(y_true_flat, y_prob_flat)


#precision, recall, thresholds = precision_recall_curve(y_true_flat, y_prob_flat)
#ap = average_precision_score(y_true_flat, y_prob_flat)

print(f"Average Precision (AP) = {ap:.4f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
plt.plot(recall, precision, linewidth=2)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve")
plt.grid(True)
plt.savefig("PRC.png", dpi=300, bbox_inches="tight")
plt.show()